In [13]:
import math
import time
from dataclasses import dataclass
from tdmclient import ClientAsync

import numpy as np
from scipy.interpolate import interp1d

# ------------------------------------------------------------------
# IMPORT YOUR CALIBRATION / SENSOR GEOMETRY
# ------------------------------------------------------------------
# If this file is inside the same package as your existing code, you
# may need to change this import to:
#   from .local_occupancy import ...
try:
    from local_occupancy import (
        sensor_measurements,
        sensor_distances,
        sensor_pos_from_center,
        sensor_angles,
    )
except ImportError:
    # Placeholder if module is not found when testing this snippet here.
    # In your real project REMOVE this whole block and keep the import above.
    sensor_measurements = np.array([0, 4000])
    sensor_distances = np.array([100.0, 2.0])
    sensor_pos_from_center = [(0.0, 0.0)] * 7
    sensor_angles = [0.0] * 7


# ------------------------------------------------------------------
# BASIC UTILITIES
# ------------------------------------------------------------------


def wrap_to_pi(angle: float) -> float:
    """Wrap any angle (rad) to (-pi, pi]."""
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


def sensor_val_to_cm_dist(val: int) -> float:
    """Convert a Thymio prox value to a distance in cm using your calibration."""
    if val == 0:
        return np.inf
    f = interp1d(sensor_measurements, sensor_distances)
    return float(f(val))


def obstacles_pos_from_sensor_vals(sensor_vals):
    """
    Convert Thymio's horizontal prox values to obstacle positions
    in the ROBOT frame, in centimeters.

    Robot frame:
      x_left  > 0  -> obstacle is on the LEFT
      x_left  < 0  -> obstacle is on the RIGHT
      y_forward > 0 -> obstacle is in FRONT
    """
    dists_cm = [sensor_val_to_cm_dist(v) for v in sensor_vals]

    obstacles = []
    for (sx_forward, sy_left), d, a in zip(sensor_pos_from_center, dists_cm, sensor_angles):
        if not np.isfinite(d):
            continue

        # In original frame (x_forward, y_left):
        obs_forward = sx_forward + d * math.cos(a)
        obs_left = sy_left + d * math.sin(a)

        # Convert to our robot frame (x_left, y_forward)
        x_left = obs_left
        y_forward = obs_forward

        obstacles.append([x_left, y_forward])

    return np.array(obstacles, dtype=float)


# ------------------------------------------------------------------
# LOCAL OCCUPANCY GRID
# ------------------------------------------------------------------


class LocalOccupancyGrid:
    """
    Robot-centred occupancy grid.
    x: left (+), right (-)
    y: forward (+), back (-)
    """

    def __init__(
        self,
        size_m: float = 0.6,
        resolution_m: float = 0.01,
        hit_inc: float = 0.4,
        decay: float = 0.95,
        robot_radius_m: float = 0.085,
    ):
        self.size_m = size_m
        self.resolution = resolution_m
        self.hit_inc = hit_inc
        self.decay = decay
        self.robot_radius_m = robot_radius_m

        self.half_size = size_m / 2.0
        self.n_cells = int(size_m / resolution_m)
        self.grid = np.zeros((self.n_cells, self.n_cells), dtype=float)

    def _point_to_index(self, x_m, y_m):
        # x_m, y_m are in robot frame (x_left, y_forward)
        if abs(x_m) > self.half_size or abs(y_m) > self.half_size:
            return None
        ix = int((x_m + self.half_size) / self.resolution)
        iy = int((y_m + self.half_size) / self.resolution)
        ix = min(max(ix, 0), self.n_cells - 1)
        iy = min(max(iy, 0), self.n_cells - 1)
        return iy, ix

    def _index_to_point(self, iy, ix):
        x_m = (ix + 0.5) * self.resolution - self.half_size
        y_m = (iy + 0.5) * self.resolution - self.half_size
        return x_m, y_m

    def update_from_sensor_vals(self, sensor_vals):
        """
        Update grid from Thymio horizontal proximity readings.
        Obstacles are inflated by robot radius.
        """
        # decay old values
        self.grid *= self.decay

        # convert current sensor readings to obstacle positions in robot frame
        obs_cm = obstacles_pos_from_sensor_vals(sensor_vals)

        inflate_cells = max(1, int(self.robot_radius_m / self.resolution))

        for ox_cm, oy_cm in obs_cm:
            if not np.isfinite(ox_cm) or not np.isfinite(oy_cm):
                continue

            x_m = ox_cm / 100.0
            y_m = oy_cm / 100.0

            idx = self._point_to_index(x_m, y_m)
            if idx is None:
                continue

            iy, ix = idx

            # Inflate in a disk around (iy, ix)
            for dy in range(-inflate_cells, inflate_cells + 1):
                for dx in range(-inflate_cells, inflate_cells + 1):
                    jy = iy + dy
                    jx = ix + dx
                    if 0 <= jy < self.n_cells and 0 <= jx < self.n_cells:
                        if dx * dx + dy * dy <= inflate_cells * inflate_cells:
                            self.grid[jy, jx] = min(
                                1.0, self.grid[jy, jx] + self.hit_inc
                            )


# ------------------------------------------------------------------
# LOCAL NAVIGATOR (PURE AVOIDANCE MODE)
# ------------------------------------------------------------------


@dataclass
class LocalNavConfig:
    occ_threshold: float = 0.3
    rep_influence_radius: float = 0.35
    k_att: float = 1.0
    k_rep: float = 0.8


class LocalNavigator:
    def __init__(self, grid: LocalOccupancyGrid, cfg: LocalNavConfig):
        self.grid = grid
        self.cfg = cfg

    def _repulsive_vector(self):
        """
        Repulsive force in ROBOT frame [x_left, y_forward].

        - Only from cells in front (y_forward > 0)
        - Weighted by occupancy
        - Biased to be more LATERAL than purely backwards
        - BOUNDED in norm so it can't completely dominate attraction
        """
        F_rep = np.zeros(2, dtype=float)
        R = self.cfg.rep_influence_radius
        eps = 1e-3

        occ_idxs = np.argwhere(self.grid.grid > self.cfg.occ_threshold)

        for iy, ix in occ_idxs:
            occ_val = float(self.grid.grid[iy, ix])  # in [0,1]
            x_m, y_m = self.grid._index_to_point(iy, ix)
            d = math.sqrt(x_m * x_m + y_m * y_m)

            # ignore behind robot
            if y_m <= 0.0:
                continue
            if d < eps or d > R:
                continue

            # --- more lateral repulsion ---
            # raw repulsion: from obstacle to robot
            lat = -x_m / d      # left/right
            back = -y_m / d     # backwards

            # bias: keep lateral strong, but reduce pure backward component
            dir_vec = np.array([lat, 0.4 * back], dtype=float)

            # re-normalise direction
            n = np.linalg.norm(dir_vec)
            if n < 1e-6:
                continue
            dir_vec /= n

            # distance-based weight
            s = max(0.0, (R - d) / R)
            s = s * s

            mag = self.cfg.k_rep * occ_val * s

            F_rep += mag * dir_vec

        # ---- FINAL SATURATION on F_rep magnitude ----
        max_F_rep = 3.0 * self.cfg.k_att  # allow stronger repulsion
        norm_rep = np.linalg.norm(F_rep)
        if norm_rep > max_F_rep:
            F_rep = F_rep * (max_F_rep / norm_rep)

        return F_rep

    def compute_avoidance_direction_robot(self, sensor_vals):
        """
        PURE LOCAL AVOIDANCE IN ROBOT FRAME.

        - Always "wants" to go forward (attraction).
        - Bends around obstacles using the repulsive field.

        Returns:
            F_att, F_rep, F_tot in robot frame [x_left, y_forward]
        """
        # Update occupancy grid from current readings
        self.grid.update_from_sensor_vals(sensor_vals)

        # Attractive: go straight ahead in robot frame
        F_att = np.array([0.0, self.cfg.k_att], dtype=float)

        # Repulsive from occupancy grid
        F_rep = self._repulsive_vector()

        # Optionally disable repulsion when nothing is close
        if sensor_vals is None or max(sensor_vals) < 300:
            F_rep = np.zeros(2, dtype=float)

        F_tot = F_att + F_rep

        # Avoid exact zero vector
        if np.linalg.norm(F_tot) < 1e-6:
            F_tot = np.array([0.0, 1.0], dtype=float)

        return F_att, F_rep, F_tot


# ------------------------------------------------------------------
# GO-TO-GOAL CONTROLLER PIECES (ONLY FOR WHEEL MAPPING)
# ------------------------------------------------------------------


@dataclass
class GoToGoalGains:
    Komega: float = 3.0
    v_max: float = 0.18
    w_max: float = 3.0


@dataclass
class ThymioKinematics:
    wheel_radius: float = 0.022
    half_axle: float = 0.0475
    motor_gain: float = 10.0
    motor_max: int = 300


class GoToGoalController:
    def __init__(self, gains: GoToGoalGains, kin: ThymioKinematics):
        self.g = gains
        self.kin = kin

    def unicycle_to_wheels(self, v, w):
        r = self.kin.wheel_radius
        L = self.kin.half_axle

        phi_L = (v / r) - (L / r) * w
        phi_R = (v / r) + (L / r) * w

        uL = int(self.kin.motor_gain * phi_L)
        uR = int(self.kin.motor_gain * phi_R)

        uL = max(-self.kin.motor_max, min(self.kin.motor_max, uL))
        uR = max(-self.kin.motor_max, min(self.kin.motor_max, uR))

        return uL, uR


# ------------------------------------------------------------------
# MAPPING FROM FORCE VECTOR -> (v, w)
# ------------------------------------------------------------------


def avoidance_unicycle_cmd(F_tot, v_nominal=0.12, Komega=3.0):
    """
    Map a desired direction vector in robot frame (x_left, y_forward)
    to unicycle (v, w).
    """
    x_left, y_forward = F_tot
    norm = float(np.linalg.norm(F_tot) + 1e-6)

    # heading angle relative to forward
    alpha = math.atan2(x_left, y_forward)  # + = turn left, - = turn right

    # how much the vector points forward (0..1)
    forward_component = max(0.0, y_forward / norm)

    # Option: don't let it go *too* slow even when sideways
    forward_component = max(0.3, forward_component)

    v = v_nominal * forward_component
    w = Komega * alpha

    return v, w


# ------------------------------------------------------------------
# SMALL DEBUG HELPERS
# ------------------------------------------------------------------


def vec_to_polar(v):
    x_left, y_forward = v
    mag = float(np.linalg.norm(v))
    ang_deg = math.degrees(math.atan2(x_left, y_forward))
    return mag, ang_deg



In [14]:
def avoidance_unicycle_cmd(F_tot, gains: GoToGoalGains, v_far: float = None, v_turn: float = 0.05):
    """
    Map a desired direction vector in ROBOT frame (x_left, y_forward)
    to unicycle (v, w) using your GoToGoalGains.

    - For small angles -> drive forward fast.
    - For medium angles -> slower forward, strong turn.
    - For large angles -> pivot in place (v ~ 0).
    """
    x_left, y_forward = F_tot
    norm = float(np.linalg.norm(F_tot) + 1e-6)

    if v_far is None:
        v_far = gains.v_max  # e.g. 0.3

    # heading angle relative to forward:
    alpha = math.atan2(x_left, y_forward)  # + = turn left, - = turn right
    angle_deg = math.degrees(alpha)

    # baseline forward component in [0, 1]
    forward_component = max(0.0, y_forward / norm)

    # --- piecewise behaviour depending on how sideways we want to go ---
    if abs(angle_deg) > 80:
        # almost sideways/backwards: pivot in place
        v = 0.0
    elif abs(angle_deg) > 40:
        # turning strongly: go slowly forward
        v = v_turn * forward_component
    else:
        # mostly forward: go faster
        v = v_far * forward_component

    # angular speed from angle
    w = gains.Komega * alpha

    # clip to controller limits
    v = max(-gains.v_max, min(gains.v_max, v))
    w = max(-gains.w_max, min(gains.w_max, w))

    return v, w



def vec_to_polar(v):
    """
    Convert a robot-frame vector [x_left, y_forward] into (magnitude, angle_deg).
    angle_deg:
      0°  -> straight ahead
      >0° -> steer left
      <0° -> steer right
    """
    x_left, y_forward = v
    mag = float(np.linalg.norm(v))
    ang_deg = math.degrees(math.atan2(x_left, y_forward))
    return mag, ang_deg


In [15]:
grid = LocalOccupancyGrid(
    size_m=0.8,
    resolution_m=0.01,
    hit_inc=0.4,
    decay=0.85,
    robot_radius_m=0.12,
)

cfg = LocalNavConfig(
    #lookahead_dist_m=0.15,      # unused here
    #max_lookahead_points=30,    # unused here
    occ_threshold=0.3,
    rep_influence_radius=0.8,
    k_att=1.2,
    k_rep=0.85,
   # virt_goal_dist_m=0.10,      # unused here
)

navigator = LocalNavigator(grid, cfg)

gains = GoToGoalGains(
    #Kv=2.0,       # unused here
    Komega=1.0,
    v_max=0.3,
    w_max=3.0,
)
kin = ThymioKinematics()
g2g = GoToGoalController(gains, kin)


In [16]:
# Thymio connection
client = ClientAsync()
node = await client.wait_for_node()
await node.lock()
await node.wait_for_variables({"prox.horizontal"})

def motors(l_speed=0, r_speed=0):
    return {
        "motor.left.target": [int(l_speed)],
        "motor.right.target": [int(r_speed)],
    }


In [6]:
async def run():
    
    try:
        while True:
            # --- get sensors from Thymio ---
            sensor_vals = list(node.v.prox.horizontal)

            # --- update local occupancy grid from sensors ---
            grid.update_from_sensor_vals(sensor_vals)

            # --- attractive force: always "go straight forward" in robot frame ---
            # robot frame: [x_left, y_forward]
            F_att = np.array([0.0, cfg.k_att], dtype=float)

            # --- repulsive force from occupancy grid ---
            F_rep = navigator._repulsive_vector()

            # optionally disable repulsion when nothing is close
            if sensor_vals is None or max(sensor_vals) < 300:
                F_rep = np.zeros(2, dtype=float)

            # --- total force in robot frame ---
            F_tot = F_att + F_rep
            if np.linalg.norm(F_tot) < 1e-6:
                # avoid zero vector: just go forward
                F_tot = np.array([0.0, 1.0], dtype=float)

            # --- map total force -> unicycle command (v,w) ---
            v, w = avoidance_unicycle_cmd(F_tot, gains=gains)

            # --- map unicycle -> wheel commands and send to Thymio ---
            uL, uR = g2g.unicycle_to_wheels(v, w)

            await node.set_variables({
                "motor.left.target":  [uL],
                "motor.right.target": [uR],
            })

            # --- DEBUG PRINTS: see what the vectors are doing ---
            mag_att, ang_att = vec_to_polar(F_att)
            mag_rep, ang_rep = vec_to_polar(F_rep)
            mag_tot, ang_tot = vec_to_polar(F_tot)

            print(
                f"sensors={sensor_vals} | "
                f"F_att mag={mag_att:.2f}, ang={ang_att:+.1f}° | "
                f"F_rep mag={mag_rep:.2f}, ang={ang_rep:+.1f}° | "
                f"F_tot mag={mag_tot:.2f}, ang={ang_tot:+.1f}° | "
                f"v={v:.3f}, w={w:.3f}, uL={uL}, uR={uR}"
            )

            # run at ~10 Hz
            await client.sleep(0.1)

    finally:
        # stop motors at the end even if something crashes
        await node.set_variables({
            "motor.left.target":  [0],
            "motor.right.target": [0],
        })
        await node.unlock()


In [17]:
def apply_side_heuristic(sensor_vals, w, min_turn=0.8, side_thr=1200):
    """
    Flip / enforce the sign of w based on which outer front sensors see an obstacle.

    Assumes Thymio prox.horizontal indices:
      0: front left
      1: front left-center
      2: front center
      3: front right-center
      4: front right
      5: rear left
      6: rear right

    - If the far LEFT side is stronger -> force a RIGHT turn (w < 0)
    - If the far RIGHT side is stronger -> force a LEFT turn (w > 0)
    """
    s = sensor_vals

    # Max of the two left-front sensors
    left_side = max(s[0], s[1])
    # Max of the two right-front sensors
    right_side = max(s[3], s[4])

    # nothing significant
    if left_side < side_thr and right_side < side_thr:
        return w

    # Decide which side dominates
    if right_side > left_side and right_side >= side_thr:
        # obstacle on RIGHT -> force LEFT turn (w > 0)
        if w <= 0:
            w = min_turn
        else:
            w = max(w, min_turn)
    elif left_side > right_side and left_side >= side_thr:
        # obstacle on LEFT -> force RIGHT turn (w < 0)
        if w >= 0:
            w = -min_turn
        else:
            w = min(w, -min_turn)

    return w


In [19]:
async def run():
    print("Starting PURE OBSTACLE AVOIDANCE test...")

    try:
        while True:
            # --- get sensors from Thymio ---
            sensor_vals = list(node.v.prox.horizontal)

            # --- compute forces in robot frame (this updates grid internally) ---
            F_att, F_rep, F_tot = navigator.compute_avoidance_direction_robot(sensor_vals)

            # --- base (v, w) from potential field ---
            v, w = avoidance_unicycle_cmd(F_tot, gains=gains)

            # --- apply 'outer sensors' heuristic on w ---
            w = apply_side_heuristic(sensor_vals, w, min_turn=1.0, side_thr=1200)

            # --- map unicycle -> wheel commands and send to Thymio ---
            uL, uR = g2g.unicycle_to_wheels(v, w)

            await node.set_variables({
                "motor.left.target":  [uL],
                "motor.right.target": [uR],
            })

            # --- Debug print ---
            mag_att, ang_att = vec_to_polar(F_att)
            mag_rep, ang_rep = vec_to_polar(F_rep)
            mag_tot, ang_tot = vec_to_polar(F_tot)

            print(
                f"sensors={sensor_vals} | "
                f"F_att mag={mag_att:.2f}, ang={ang_att:+.1f}° | "
                f"F_rep mag={mag_rep:.2f}, ang={ang_rep:+.1f}° | "
                f"F_tot mag={mag_tot:.2f}, ang={ang_tot:+.1f}° | "
                f"v={v:.3f}, w={w:.3f}, uL={uL}, uR={uR}"
            )

            await client.sleep(0.1)

    finally:
        await node.set_variables({
            "motor.left.target":  [0],
            "motor.right.target": [0],
        })
        await node.unlock()


In [20]:
await run()


Starting PURE OBSTACLE AVOIDANCE test...
sensors=[0, 0, 3593, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=3.60, ang=-105.6° | F_tot mag=3.48, ang=-86.2° | v=0.000, w=-1.504, uL=32, uR=-32
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.300, w=0.000, uL=136, uR=136
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.300, w=0.000, uL=136, uR=136
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.300, w=0.000, uL=136, uR=136
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.300, w=0.000, uL=136, uR=136
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.300, w=0.000, uL=136, uR=136
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° |

CancelledError: 